# Random Forests and Extra Trees

## 1. Definition
A Random Forest trains many decision trees on bootstrap samples and randomizes the subset of candidate features considered at each split. Extra Trees (Extremely Randomized Trees) inject additional randomness into split thresholds and may use the full dataset rather than bootstrap samples by default. Both aggregate tree predictions to reduce variance.

## 2. Why feature randomness matters
If every tree is dominated by the same strong feature, trees can be highly correlated. Limiting candidate features makes trees less alike; averaging then has more opportunity to reduce variance. More randomness can increase individual-tree bias, so there is a bias-variance trade-off.

## 3. Objective and assumptions

A regression tree typically chooses splits that reduce within-node squared error; a classification tree often uses Gini impurity or entropy. Random Forest does not optimize one simple global differentiable objective. Instead, it aggregates many randomized greedy trees.

A forest is not guaranteed to beat a single tree on every dataset. It is often robust to nonlinear feature interactions, but may extrapolate poorly in regression and can be large at inference. Feature scaling is usually unnecessary for axis-aligned trees.

In [1]:
import time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, RandomForestRegressor, ExtraTreesRegressor
from sklearn.metrics import accuracy_score, classification_report, mean_squared_error, r2_score

SEED = 42
X, y = make_classification(n_samples=1600, n_features=20, n_informative=8, n_redundant=5, class_sep=0.9, random_state=SEED)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)
print(X_train.shape)


(1200, 20)


## 4. Forest mechanics from scratch

This implementation demonstrates bootstrap sampling, random feature subsets chosen once per tree, and voting. A production Random Forest typically samples features at every split, not just once per tree; this educational version is closer to a random-subspace bagged forest. The distinction matters.

In [2]:
def fit_random_subspace_forest(X, y, n_estimators=50, max_features=5, max_depth=None, seed=42):
    X = np.asarray(X); y = np.asarray(y)
    if X.ndim != 2 or y.ndim != 1 or len(X) != len(y) or len(X) == 0:
        raise ValueError("X must be non-empty 2D and y matching 1D")
    if not 1 <= max_features <= X.shape[1] or n_estimators < 1:
        raise ValueError("Invalid n_estimators or max_features")
    rng = np.random.default_rng(seed)
    models = []
    for _ in range(n_estimators):
        rows = rng.choice(len(X), size=len(X), replace=True)
        cols = np.sort(rng.choice(X.shape[1], size=max_features, replace=False))
        tree = DecisionTreeClassifier(max_depth=max_depth, random_state=int(rng.integers(0, 2**31-1)))
        tree.fit(X[rows][:, cols], y[rows])
        models.append((tree, cols))
    return models

def predict_random_subspace_forest(models, X):
    preds = np.stack([tree.predict(np.asarray(X)[:, cols]) for tree, cols in models])
    # Classification labels here are numeric; mode via bincount for binary/multiclass nonnegative labels.
    out = []
    for j in range(preds.shape[1]):
        values, counts = np.unique(preds[:, j], return_counts=True)
        out.append(values[np.argmax(counts)])
    return np.asarray(out)

scratch_forest = fit_random_subspace_forest(X_train, y_train, n_estimators=70, max_features=6, max_depth=None, seed=SEED)
pred_scratch = predict_random_subspace_forest(scratch_forest, X_test)
print("Scratch random-subspace forest accuracy:", accuracy_score(y_test, pred_scratch))


Scratch random-subspace forest accuracy: 0.835


## 5. Compare with Random Forest and Extra Trees

`max_features` controls the number of candidate features per split. The values and defaults differ by task and estimator; inspect documentation for the installed version when relying on defaults.

In [3]:
rf = RandomForestClassifier(n_estimators=200, max_features="sqrt", min_samples_leaf=1,
                            random_state=SEED, n_jobs=-1).fit(X_train, y_train)
et = ExtraTreesClassifier(n_estimators=200, max_features="sqrt", min_samples_leaf=1,
                          random_state=SEED, n_jobs=-1).fit(X_train, y_train)
for name, model in [("Scratch random-subspace", None), ("Random Forest", rf), ("Extra Trees", et)]:
    pred = pred_scratch if model is None else model.predict(X_test)
    print(name, "accuracy:", round(accuracy_score(y_test, pred), 4))


Scratch random-subspace accuracy: 0.835
Random Forest accuracy: 0.85
Extra Trees accuracy: 0.875


## 6. Hyperparameters and diagnostics

- `n_estimators`: more trees reduce Monte Carlo variability, with diminishing returns.
- `max_features`: lower values tend to decorrelate trees but can raise bias.
- `max_depth`, `min_samples_leaf`, `min_samples_split`: control tree complexity and overfitting.
- `bootstrap`: whether each tree gets a bootstrap sample.
- `class_weight`: can help class imbalance, but evaluate per-class metrics.
- Out-of-bag (OOB) estimates are possible with bootstrap sampling, but are not a replacement for a final held-out evaluation.
- Feature importance is not causality; impurity-based importance can favor high-cardinality features. Consider permutation importance.

## 7. Complexity and scaling

For n samples, d features, T trees and average tree depth h, a rough prediction cost is O(T·h) per sample for a decision tree traversal, with implementation-dependent overhead. Training often grows roughly with T and the number of candidate split evaluations; exact complexity depends heavily on tree depth, split strategy and feature count.

For large datasets, constrain depth/leaves, use `n_jobs` sensibly, monitor serialized model size and prediction latency, and compare histogram-based boosting if training time is a concern. Tree models usually need no scaling. GPU support is library-specific; scikit-learn's RandomForest and ExtraTrees are CPU-based.

In [4]:
# Compare training time and inference batch time.
for name, factory in [
    ("RF 50 trees", lambda: RandomForestClassifier(n_estimators=50, random_state=SEED, n_jobs=-1)),
    ("ExtraTrees 50 trees", lambda: ExtraTreesClassifier(n_estimators=50, random_state=SEED, n_jobs=-1))
]:
    model = factory()
    t0 = time.perf_counter(); model.fit(X_train, y_train); train_s = time.perf_counter()-t0
    t0 = time.perf_counter(); model.predict(X_test); infer_s = time.perf_counter()-t0
    print(name, {"train_seconds": round(train_s, 3), "batch_predict_seconds": round(infer_s, 4)})


RF 50 trees {'train_seconds': 0.084, 'batch_predict_seconds': 0.0125}


ExtraTrees 50 trees {'train_seconds': 0.064, 'batch_predict_seconds': 0.0126}


## 8. Correctness tests and synthesis

The scratch forest is intentionally a random-subspace ensemble, not a full reproduction of Random Forest's per-split feature sampling. It demonstrates the ensemble principle. Use scikit-learn's tested implementation for practical work unless you need a custom variant.

In [5]:
assert pred_scratch.shape == y_test.shape
assert np.isfinite(pred_scratch).all()
assert set(np.unique(pred_scratch)).issubset(set(np.unique(y_train)))
try:
    fit_random_subspace_forest(np.array([1,2,3]), np.array([0,1,0]))
except ValueError:
    pass
else:
    raise AssertionError("Expected invalid X shape to raise")
print("All forest checks passed.")


All forest checks passed.
